# Results 6 — Genetic evidence and therapeutic success

Enrichment of GWAS genetic support among approved target-indication pairs, how it varies with
variant frequency, effect size, protein-altering status and gene-level pleiotropy, and the
combined criterion that performs best.

| file | panel |
| --- | --- |
| `temporal_drug_enrichment_full_chembl.csv` | Figure 5a |
| `drug_enrichment_subsets_vs_full_l2g.csv` | Figure 5b |
| `df_for_enrichment_regression.csv` | Figure 5c |

All strata are computed from the one pair-level table `ti_pairs_chembl`, so no definition can
drift between rows. Odds ratios are Fisher's exact; relative success is a risk ratio.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from manuscript_methods import paper
from manuscript_methods.enrichment import bh_plain, contrast, or_rs, support_mask

numbers = {}
master = pd.read_parquet(paper.derived("ti_pairs_chembl"))
approved = master["approved"].to_numpy().astype(bool)
print("target-indication pairs:", len(master), "| approved:", int(approved.sum()))

## Overall enrichment

In [ ]:
overall = or_rs(support_mask(master), master["approved"])
numbers["R6.01"] = overall["yes_evid-high_clinphase"]
numbers["R6.02"] = round(overall["odds_ratio"], 2)
numbers["R6.03"] = round(overall["relative_success"], 2)
print({k: numbers[k] for k in ["R6.01", "R6.02", "R6.03"]})
print("P(OR) = %.2e  P(RS) = %.2e" % (overall["p_value"], overall["rs_p_value"]))

## Figure 5b — enrichment by class of genetic support

Each group contrasts a restricted definition of support against the complement of that
restriction among supported pairs. Rare means the supporting credible set has MAF below 0.01,
large effect means a rescaled absolute effect above 0.5, PAV means the credible set contains a
protein-altering variant.

In [ ]:
STRATA = {
    "PAV": (
        ("PAV", lambda d: d["score_all"].notna() & (d["max_vep"] == 1)),
        ("non-PAV", lambda d: d["score_all"].notna() & (d["max_vep"] != 1)),
    ),
    "effect size": (
        ("large effect", lambda d: d["score_all"].notna() & (d["max_beta"] > 0.5)),
        ("small effect", lambda d: d["score_all"].notna() & (d["max_beta"] <= 0.5)),
    ),
    "variant frequency": (
        ("rare", lambda d: d["score_all"].notna() & (d["min_maf"] < 0.01)),
        ("common", lambda d: d["score_all"].notna() & (d["min_maf"] >= 0.01)),
    ),
    "gPS": (
        ("gPS<=5", lambda d: support_mask(d, gps_min=1, gps_max=5)),
        ("gPS>=10", lambda d: support_mask(d, gps_min=10)),
    ),
    "therapeutic areas": (
        ("TAs=1", lambda d: support_mask(d, ta_min=1, ta_max=1)),
        ("TAs>=6", lambda d: support_mask(d, ta_min=6)),
    ),
}


def stratum_rows(group, first, second):
    \"\"\"Two forest rows and the within-group contrast between them.\"\"\"
    (first_label, first_mask), (second_label, second_mask) = first, second
    masks = {first_label: first_mask(master), second_label: second_mask(master)}
    rows = [{"group": group, "stratum": label, **or_rs(mask, master["approved"])} for label, mask in masks.items()]
    counts = {
        "x_low": int((masks[first_label] & approved).sum()),
        "n_low": int((masks[first_label] & ~approved).sum()),
        "x_high": int((masks[second_label] & approved).sum()),
        "n_high": int((masks[second_label] & ~approved).sum()),
    }
    test = {"group": group, "low": first_label, "high": second_label, **counts, **contrast(**counts)}
    return rows, test


forest, tests = [], []
for group, (first, second) in STRATA.items():
    rows, test = stratum_rows(group, first, second)
    forest += rows
    tests.append(test)

forest = pd.DataFrame(forest)
tests = pd.DataFrame(tests)
tests["fdr"] = bh_plain(tests["p_value"])
forest.to_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"), index=False)
tests.to_csv(paper.derived("figure_5b_contrasts.csv"), index=False)

print(forest[["group", "stratum", "odds_ratio", "ci_low", "ci_high", "yes_evid-high_clinphase"]].round(3).to_string(index=False))
print()
print(tests[["group", "low", "high", "p_value", "fdr"]].round(5).to_string(index=False))

In [ ]:
by_stratum = forest.set_index("stratum")["odds_ratio"]
by_group = tests.set_index("group")

numbers["R6.05"] = round(float(by_stratum["rare"]), 1)
numbers["R6.06"] = round(float(by_stratum["common"]), 1)
numbers["R6.07"] = round(float(by_group.loc["variant frequency", "p_value"]), 4)
numbers["R6.13"] = round(float(by_stratum["PAV"]), 1)
numbers["R6.14"] = round(float(by_stratum["non-PAV"]), 1)
numbers["R6.15"] = round(float(by_group.loc["PAV", "p_value"]), 4)
numbers["R6.17"] = round(float(by_stratum["large effect"]), 1)
numbers["R6.18"] = round(float(by_stratum["small effect"]), 1)
numbers["R6.19"] = round(float(by_stratum["gPS<=5"]), 1)
numbers["R6.20"] = round(float(by_stratum["gPS>=10"]), 1)
numbers["R6.21"] = round(float(by_group.loc["gPS", "p_value"]), 3)
numbers["R6.22"] = round(float(by_stratum["TAs=1"]), 1)
numbers["R6.23"] = round(float(by_stratum["TAs>=6"]), 1)
numbers["R6.24"] = round(float(by_group.loc["therapeutic areas", "p_value"]), 2)
print({k: numbers[k] for k in sorted(numbers) if k >= "R6.05"})

## Rare disease resources and gene-based tests

Each resource is propagated through the ontology and tested against the same ChEMBL pairs, using
the library implementation the published analysis used.

In [ ]:
from gentropy.common.session import Session
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

disease_index = session.spark.read.parquet(paper.release("disease") + "/disease.parquet")
chembl_evidence = session.spark.read.parquet(paper.release("evidence") + "/sourceId=chembl")
all_evidence = session.spark.read.parquet(paper.release("evidence"))
combined = session.spark.read.parquet(paper.baseline("combined_evidence_with_measurements"))
print(combined.groupBy("source").count().toPandas().to_string(index=False))

In [ ]:
def enrichment_of(evidence, label, threshold=0.0):
    \"\"\"Odds ratio and relative success for one evidence source against the ChEMBL pairs.\"\"\"
    table = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_evidence,
        indirect_assoc_score_thr=threshold,
        efo_ancestors_to_remove=["MONDO_0045024"],
    )
    table["datasource"] = label
    return table


def platform_evidence(datasources, score=0.75):
    \"\"\"Release evidence rows from the given datasources above a score threshold.\"\"\"
    return (
        all_evidence.filter(f.col("score") >= score)
        .filter(f.col("datasourceId").isin(datasources))
        .drop("resourceScore")
        .withColumn("resourceScore", f.lit(1.0))
    )


sources = [
    ("OMIM", combined.filter(f.col("source") == "omim")),
    ("Orphanet", combined.filter(f.col("source") == "orphanet")),
    ("Gene-based tests", combined.filter(f.col("source") == "gene_burden")),
    ("ClinVar/ClinGen", platform_evidence(["eva", "clingen"])),
    ("UniProt", platform_evidence(["uniprot_variants", "uniprot_literature"])),
    ("The Genomics England PanelApp", platform_evidence(["genomics_england"])),
]

resources = pd.concat([enrichment_of(evidence, label) for label, evidence in sources], ignore_index=True)
resources.to_csv(paper.derived("drug_enrichment_other_resources.csv"), index=False)
approved_rows = resources[resources["clinicalPhase"] == "4+"].set_index("datasource")
print(approved_rows[["odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3).to_string())

In [ ]:
for key, label in [
    ("R6.08", "Orphanet"),
    ("R6.09", "OMIM"),
    ("R6.10", "ClinVar/ClinGen"),
    ("R6.11", "UniProt"),
    ("R6.12", "The Genomics England PanelApp"),
    ("R6.16", "Gene-based tests"),
]:
    numbers[key] = round(float(approved_rows.loc[label, "odds_ratio"]), 1)
print({k: numbers[k] for k in ["R6.08", "R6.09", "R6.10", "R6.11", "R6.12", "R6.16"]})

## Figure 5c — probability of success against pleiotropy

The regression frame is the pair-level table with the propagated variant features. Non-linearity
is tested by likelihood ratio against a linear pleiotropy term.

In [ ]:
regression = master[
    [
        "targetId",
        "diseaseId",
        "score_all",
        "max_beta",
        "min_maf",
        "max_vep",
        "maxClinicalPhase",
        "uniqueDiseases",
        "uniqueTherapeuticAreas",
    ]
].copy()
regression = regression.rename(columns={"score_all": "indirect_assoc_score"}).fillna(
    {
        "indirect_assoc_score": 0.0,
        "max_beta": 0.0,
        "min_maf": 0.0,
        "max_vep": 0.0,
        "uniqueDiseases": 0.0,
        "uniqueTherapeuticAreas": 0.0,
    }
)
regression["outcome"] = (regression["maxClinicalPhase"] >= 4).astype(int)
regression["geneticSupport"] = (regression["indirect_assoc_score"] >= 0.1).astype(int)
regression.to_csv(paper.derived("df_for_enrichment_regression.csv"), index=False)
print(regression.shape)
regression.head(3)

In [ ]:
def nonlinearity(pleiotropy):
    \"\"\"Likelihood ratio test of a log pleiotropy term against a linear one.\"\"\"
    df = regression.copy()
    df["pleiotropy"] = df[pleiotropy]
    df["log_pleiotropy"] = np.log1p(df["pleiotropy"])
    baseline = smf.logit("outcome ~ geneticSupport", data=df).fit(disp=False)
    linear = smf.logit("outcome ~ geneticSupport + pleiotropy", data=df).fit(disp=False)
    log_model = smf.logit("outcome ~ geneticSupport + pleiotropy + log_pleiotropy", data=df).fit(disp=False)
    return {
        "pleiotropy": pleiotropy,
        "llr_vs_baseline": float(log_model.llr_pvalue),
        "p_log_term": float(log_model.pvalues["log_pleiotropy"]),
        "p_linear_term": float(linear.pvalues["pleiotropy"]),
        "llf_baseline": float(baseline.llf),
        "llf_linear": float(linear.llf),
        "llf_log": float(log_model.llf),
    }


tests_nonlinear = pd.DataFrame([nonlinearity("uniqueDiseases"), nonlinearity("uniqueTherapeuticAreas")])
tests_nonlinear.to_csv(paper.derived("figure_5c_nonlinearity.csv"), index=False)
tests_nonlinear

## High pleiotropy against no genetic support

Highly pleiotropic supported pairs are still more successful than clinical candidates with no
GWAS support for that pair, which is the reference the odds ratio below is taken against.

In [ ]:
high_pleiotropy = support_mask(master, gps_min=10)
unsupported = ~support_mask(master)
subset = pd.DataFrame({"outcome": master["approved"], "support": np.where(high_pleiotropy, 1, 0)})[
    (high_pleiotropy | unsupported).to_numpy()
]
model = smf.logit("outcome ~ support", data=subset).fit(disp=False)
print("OR high pleiotropy vs no support: %.3f  P = %.2e" % (np.exp(model.params["support"]), model.pvalues["support"]))
numbers["R6.25"] = round(float(np.exp(model.params["support"])), 2)

## The combined criterion

Protein-altering support with genetic support in two to five therapeutic areas. The definition
comes from the two observations above, not from a search over thresholds.

In [ ]:
strict_mask = support_mask(master, pav=True, ta_min=2, ta_max=5)
strict = or_rs(strict_mask, master["approved"])
numbers["R6.28"] = round(strict["odds_ratio"], 1)
numbers["R6.29"] = round(strict["relative_success"], 1)
numbers["R6.30"] = strict["yes_evid-high_clinphase"]
numbers["R6.31"] = round(100 * strict["yes_evid-high_clinphase"] / numbers["R6.01"], 1)
print({k: numbers[k] for k in ["R6.28", "R6.29", "R6.30", "R6.31"]})
print(
    "2x2:",
    [
        [strict["no_evid-low_clinphase"], strict["no_evid-high_clinphase"]],
        [strict["yes_evid-low_clinphase"], strict["yes_evid-high_clinphase"]],
    ],
)

In [ ]:
# Supplementary Table 7: the gene-disease associations meeting the criterion.
genes = pd.read_parquet(paper.derived("gene_table"))[["geneId", "uniqueTherapeuticAreas"]]
window = set(genes.loc[genes["uniqueTherapeuticAreas"].between(2, 5), "geneId"])

l2g = pd.read_parquet(paper.derived("prioritised_genes_diseases"))[
    ["geneId", "diseaseIds", "VEP", "variantId", "score"]
]
pav_rows = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(window)].explode("diseaseIds")
associations = pav_rows[["geneId", "diseaseIds"]].dropna().drop_duplicates()
numbers["R6.27"] = len(associations)
associations.to_csv(paper.derived("st7_pav_gene_disease_pairs.csv"), index=False)
print("gene-disease associations meeting the criterion:", numbers["R6.27"])

## Previously approved targets

Whether a target already approved for another indication is more likely to succeed.

In [ ]:
supported = master[support_mask(master)].copy()
approved_targets = set(master.loc[master["approved"] == 1, "targetId"])
supported["previously_approved"] = supported["targetId"].isin(approved_targets).astype(int)
repurposing = or_rs(supported["previously_approved"].astype(bool), supported["approved"])
numbers["R6.26"] = round(repurposing["odds_ratio"], 2)
print("OR for a previously approved target:", numbers["R6.26"])

## Figure 5a — enrichment over time

For each year, only the credible sets published up to that year contribute genetic support, and
the enrichment is recomputed against the full ChEMBL pair set.

In [ ]:
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus

sl = StudyLocus.from_parquet(session, paper.release("credible_set"))
si = StudyIndex.from_parquet(session, paper.release("study"))
l2g_spark = session.spark.read.parquet(paper.derived("prioritised_genes_diseases"))

yearly = []
for year in range(2008, 2026):
    evidence = chemblDrugEnrichment.to_disease_target_evidence(
        table_with_score=l2g_spark.filter(f.col("year") <= year).drop("diseaseIds"),
        score_column="score",
        datasource_id="l2g",
        study_locus=sl,
        study_index=si,
        min_score=0.1,
    )
    table = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
        evid=evidence,
        disease_index_orig=disease_index,
        chembl_orig=chembl_evidence,
        indirect_assoc_score_thr=0.1,
        efo_ancestors_to_remove=["MONDO_0045024"],
    )
    table["datasource"] = str(year)
    table["drugsource"] = "full_chembl"
    yearly.append(table)
    print(year, "done")

temporal = pd.concat(yearly, ignore_index=True)
temporal.to_csv(paper.derived("temporal_drug_enrichment_full_chembl.csv"), index=False)
temporal[temporal["clinicalPhase"] == "4+"][["datasource", "odds_ratio", "yes_evid-high_clinphase"]].round(3)

## Numbers

In [ ]:
print(paper.save_results("therapeutic_success", numbers))
pd.Series(numbers).to_frame("computed")